# Conversation Window Memory

Neste notebook vamos estudar **Conversation Window Memory**, usando:

- Python
- LangChain
- Ollama
- Qwen 3 4B
- `ConversationBufferWindowMemory`

A ideia principal é entender como limitar a quantidade de histórico que a aplicação envia para o modelo.

> **Importante:** `ConversationBufferWindowMemory` é uma API clássica/legada do LangChain. Como a aula utiliza essa abordagem, vamos reproduzi-la com `langchain_classic`.


## 1. Por que usar uma janela de memória?

Na aula anterior usamos `ConversationBufferMemory`.

Ela mantém o histórico completo da conversa:

```text
Mensagem 1
Resposta 1
Mensagem 2
Resposta 2
Mensagem 3
Resposta 3
...
```

Em uma conversa muito longa, isso pode gerar um histórico cada vez maior.

A **Conversation Window Memory** resolve isso mantendo somente uma quantidade limitada das interações mais recentes.

Por exemplo, com:

```python
k=2
```

a memória mantém somente as **2 interações mais recentes**.


## 2. Importando as classes

Como estamos usando a API clássica do LangChain, os componentes de memória e `ConversationChain` vêm de `langchain_classic`.


In [ ]:
from langchain_ollama import ChatOllama
from langchain_classic.memory import ConversationBufferWindowMemory
from langchain_classic.chains import ConversationChain


## 3. Configurando o Ollama

Assim como nos exemplos anteriores, o modelo roda localmente através do Ollama.

Não estamos utilizando a API paga da OpenAI.


In [ ]:
llm = ChatOllama(
    model="qwen3:4b",
    base_url="http://localhost:11434"
)


## 4. Criando a memória com janela

Aqui está a principal diferença:

```python
memory = ConversationBufferWindowMemory(
    k=2,
    return_messages=True
)
```

O parâmetro `k` define o tamanho da janela.

### Exemplo

Se `k=2`, a aplicação mantém as duas interações mais recentes:

```text
Interação 1 → sai da janela quando necessário
Interação 2 → permanece
Interação 3 → permanece
```

Quando uma nova interação entra, a mais antiga é descartada.

Isso é útil para evitar que o histórico cresça indefinidamente.


In [ ]:
memory = ConversationBufferWindowMemory(
    k=2,
    return_messages=True
)


## 5. Criando a ConversationChain

A `ConversationChain` conecta o modelo à memória.

A partir desse momento, cada chamada de `invoke()` pode utilizar o histórico que ainda estiver dentro da janela.


In [ ]:
conversation = ConversationChain(
    llm=llm,
    memory=memory,
    verbose=True
)


## 6. Primeira interação

Vamos informar nosso nome e o que estamos estudando.


In [ ]:
resposta = conversation.invoke(
    "Meu nome é Felipe e estou estudando inteligência artificial."
)

print(resposta["response"])


## 7. Segunda interação

Agora adicionamos outra informação.

Como `k=2`, as duas primeiras interações ainda cabem na janela.


In [ ]:
resposta = conversation.invoke(
    "Estou estudando Python e LangChain."
)

print(resposta["response"])


## 8. Terceira interação

Agora adicionamos uma terceira interação.

Como nossa janela possui tamanho 2, a memória começa a descartar as interações mais antigas.


In [ ]:
resposta = conversation.invoke(
    "Também estou aprendendo a usar o Ollama."
)

print(resposta["response"])


## 9. Quarta interação

Depois de mais uma mensagem, somente as interações mais recentes permanecem na janela.

Isso significa que informações muito antigas podem deixar de estar disponíveis para o modelo.

Essa é a principal diferença em relação à `ConversationBufferMemory`.


In [ ]:
resposta = conversation.invoke(
    "Qual tecnologia local estou usando para executar o modelo?"
)

print(resposta["response"])


## 10. Visualizando a memória

Podemos inspecionar o conteúdo que atualmente está armazenado na memória.


In [ ]:
print(memory.load_memory_variables({}))


## 11. Buffer Memory vs Window Memory

### `ConversationBufferMemory`

Mantém todo o histórico:

```text
Mensagem 1
Mensagem 2
Mensagem 3
Mensagem 4
Mensagem 5
...
```

### `ConversationBufferWindowMemory`

Mantém somente uma janela:

```text
          ┌───────────────────────┐
Histórico │ M1 M2 M3 M4 M5 M6 M7 │
          └───────────┬───────────┘
                      ↓
              janela mais recente
                  M6 M7
```

Com `k=2`, somente as duas interações mais recentes ficam disponíveis.

### Trade-off

Quanto menor a janela:

- menor quantidade de histórico;
- menor contexto disponível;
- potencialmente menor custo de processamento;
- menor capacidade de lembrar informações antigas.

Quanto maior a janela:

- mais contexto;
- mais informações disponíveis;
- histórico maior sendo enviado ao modelo.


## 12. Um detalhe importante sobre `k`

O `k` representa a quantidade de **interações da conversa** mantidas pela memória, e não simplesmente a quantidade de mensagens individuais que você escreveu.

Uma interação normalmente envolve:

```text
Human → pergunta
AI    → resposta
```

Portanto, ao estudar essa classe, pense em `k` como o tamanho da janela de turnos da conversa.


## 13. Memória não significa que o modelo aprendeu

É importante separar duas coisas.

A memória da aplicação:

```text
Aplicação
   ↓
Histórico
   ↓
Prompt enviado ao modelo
   ↓
Qwen
```

não significa que o Qwen modificou seus parâmetros ou aprendeu permanentemente uma nova informação.

A aplicação apenas mantém contexto e o reutiliza nas próximas chamadas.

Se a memória for apagada, aquele contexto deixa de ser enviado.


## 14. Código completo

Abaixo está o exemplo completo utilizado neste estudo.


In [ ]:
from langchain_ollama import ChatOllama
from langchain_classic.memory import ConversationBufferWindowMemory
from langchain_classic.chains import ConversationChain


# 1. Configurando o modelo local do Ollama
llm = ChatOllama(
    model="qwen3:4b",
    base_url="http://localhost:11434"
)


# 2. Criando uma memória com janela de 2 interações
#
# k=2 significa que a memória mantém as 2 interações mais recentes
# da conversa.
memory = ConversationBufferWindowMemory(
    k=2,
    return_messages=True
)


# 3. Criando a conversa usando a memória
conversation = ConversationChain(
    llm=llm,
    memory=memory,
    verbose=True
)


# 4. Primeira interação
resposta = conversation.invoke(
    "Meu nome é Felipe e estou estudando inteligência artificial."
)

print("Resposta 1:")
print(resposta["response"])


# 5. Segunda interação
resposta = conversation.invoke(
    "Estou estudando Python e LangChain."
)

print("\nResposta 2:")
print(resposta["response"])


# 6. Terceira interação
resposta = conversation.invoke(
    "Também estou aprendendo a usar o Ollama."
)

print("\nResposta 3:")
print(resposta["response"])


# 7. Quarta interação
#
# Como k=2, as interações mais antigas começam a sair da janela.
resposta = conversation.invoke(
    "Qual tecnologia local estou usando para executar o modelo?"
)

print("\nResposta 4:")
print(resposta["response"])


# 8. Visualizando o conteúdo atual da memória
print("\nMemória atual:")
print(memory.load_memory_variables({}))
